# TextTrace — Final ML Experiment Pipeline

**PRG4206 Machine Learning Coursework #2**

This notebook is the Jupyter version of the final TextTrace ML script.

Every major section prints a clear `[OK]`/`[WARNING]`/`[SKIP]` message, so a code cell does not look empty after it runs.

The ML stage produces:
- model comparison metrics
- validation-only ensemble selection
- final held-out test metrics
- confusion matrix and ROC charts
- XAI outputs
- error analysis
- RAID attack robustness
- adaptive hard-example experiment
- saved models and reproducibility metadata

AI likelihood and source similarity are separate. The Streamlit stage will add similarity analysis later.


In [1]:
"""
TextTrace — Final ML Experiment Pipeline
PRG4206 Machine Learning Coursework #2
INTI International University | Session Aug 2026

Purpose
-------
This is the cleaned, final training/evaluation script for the ML stage of
TextTrace. It is designed to produce the evidence needed for the later
Streamlit system and the coursework paper.

It intentionally separates:
1. AI-generated-text detection
2. Similarity analysis (implemented later in Streamlit)

The script does NOT call a score "AI plagiarism". AI likelihood and source
similarity are separate results.

Current primary models:
- TF-IDF + Logistic Regression
- TF-IDF + Calibrated Linear SVM
- XGBoost + linguistic/style features
- DistilRoBERTa
- Validation-tuned ensemble

Optional:
- DeBERTa is left as an explicit future/extension switch because it was part
  of the literature plan but is not part of the current validated result set.

Expected local project root:
    E:\\TexTrace1

The script prefers:
    E:\\TexTrace1\\data\\processed\\raid_500k.csv

If that file is not found, it can fall back to a streaming RAID sampler.
"""

'\nTextTrace — Final ML Experiment Pipeline\nPRG4206 Machine Learning Coursework #2\nINTI International University | Session Aug 2026\n\nPurpose\n-------\nThis is the cleaned, final training/evaluation script for the ML stage of\nTextTrace. It is designed to produce the evidence needed for the later\nStreamlit system and the coursework paper.\n\nIt intentionally separates:\n1. AI-generated-text detection\n2. Similarity analysis (implemented later in Streamlit)\n\nThe script does NOT call a score "AI plagiarism". AI likelihood and source\nsimilarity are separate results.\n\nCurrent primary models:\n- TF-IDF + Logistic Regression\n- TF-IDF + Calibrated Linear SVM\n- XGBoost + linguistic/style features\n- DistilRoBERTa\n- Validation-tuned ensemble\n\nOptional:\n- DeBERTa is left as an explicit future/extension switch because it was part\n  of the literature plan but is not part of the current validated result set.\n\nExpected local project root:\n    E:\\TexTrace1\n\nThe script prefers:\n

## 0. CONFIGURATION

In [2]:
import os
import re
import json
import random
import time
import warnings
from pathlib import Path

SEED = 42
random.seed(SEED)

PROJECT_ROOT = Path(
    os.environ.get("TEXTTRACE_ROOT", r"E:\TexTrace1")
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHARTS_DIR = REPORTS_DIR / "charts"

for folder in [PROCESSED_DIR, MODELS_DIR, REPORTS_DIR, CHARTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_CSV = PROCESSED_DIR / "raid_500k.csv"

# Fallback only if the local 500K file is missing.
FALLBACK_SAMPLE_SIZE = 500_000

# DistilRoBERTa was trained on a controlled 50K subset in the current
# validated experiment. Keep this fixed for reproducibility.
TRANSFORMER_MODEL = "distilroberta-base"
TRANSFORMER_TRAIN_LIMIT = 50_000
MAX_LENGTH = 512
TRANSFORMER_EPOCHS = 2
TRANSFORMER_BATCH_SIZE = 4
TRANSFORMER_GRAD_ACCUMULATION = 4
TRANSFORMER_LR = 2e-5

# Set True only when you intentionally want to run the optional extension.
ENABLE_DEBERTA = False

EXPECTED_DOMAINS = [
    "abstracts", "books", "news", "poetry",
    "recipes", "reddit", "reviews", "wiki"
]

print("=" * 78)
print("TEXTTRACE — FINAL ML EXPERIMENT PIPELINE")
print("=" * 78)
print("[OK] Project root:", PROJECT_ROOT)
print("[OK] Reports:", REPORTS_DIR)
print("[OK] Charts:", CHARTS_DIR)
print("[OK] Seed:", SEED)

TEXTTRACE — FINAL ML EXPERIMENT PIPELINE
[OK] Project root: E:\TexTrace1
[OK] Reports: E:\TexTrace1\reports
[OK] Charts: E:\TexTrace1\reports\charts
[OK] Seed: 42


## 1. IMPORTS + PACKAGE CHECK

In [3]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

np.random.seed(SEED)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("[OK] XGBoost:", xgb.__version__)
except Exception as exc:
    XGB_AVAILABLE = False
    print("[WARNING] XGBoost unavailable:", exc)

try:
    import torch
    TORCH_AVAILABLE = True
    print("[OK] PyTorch:", torch.__version__)
    print("[OK] CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("[OK] GPU:", torch.cuda.get_device_name(0))
        print("[OK] CUDA:", torch.version.cuda)
except Exception as exc:
    TORCH_AVAILABLE = False
    print("[WARNING] PyTorch unavailable:", exc)

warnings.filterwarnings("ignore")

[OK] XGBoost: 3.2.0
[OK] PyTorch: 2.11.0+cu128
[OK] CUDA available: True
[OK] GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[OK] CUDA: 12.8


## 2. DATA LOADING

In [4]:
def clean_text(text):
    """Conservative cleaning; preserve punctuation/style information."""
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def raid_label(model):
    return 0 if model == "human" else 1


def collect_raid_sample(sample_size=500_000, seed=42):
    """
    Memory-safe fallback sampler for the RAID HF training stream.

    The local 500K CSV is preferred. This function is only used if it is
    missing. It balances the eight main English RAID domains by Human/AI.
    """
    from datasets import load_dataset

    target = sample_size // (len(EXPECTED_DOMAINS) * 2)
    buckets = {
        (domain, label): []
        for domain in EXPECTED_DOMAINS
        for label in [0, 1]
    }

    stream = load_dataset(
        "liamdugan/raid",
        "raid",
        split="train",
        streaming=True,
    )

    scanned = 0
    for record in stream:
        scanned += 1
        domain = record.get("domain")
        model = record.get("model")
        text = record.get("generation")

        if domain not in EXPECTED_DOMAINS:
            continue
        if not isinstance(text, str):
            continue

        label = raid_label(model)
        bucket = buckets[(domain, label)]

        # Keep a small reservoir inside each bucket.
        if len(bucket) < target:
            bucket.append(record)
        else:
            j = random.randint(0, scanned - 1)
            if j < target:
                bucket[j] = record

        if all(len(v) >= target for v in buckets.values()):
            break

    rows = []
    for values in buckets.values():
        rows.extend(values)

    rng = random.Random(seed)
    rng.shuffle(rows)

    df = pd.DataFrame(rows)
    print("[OK] Fallback RAID stream sample collected.")
    print("[OK] Records scanned:", f"{scanned:,}")
    return df


if INPUT_CSV.exists():
    print("[DATA] Loading local 500K dataset:", INPUT_CSV)
    df = pd.read_csv(INPUT_CSV)
else:
    print("[DATA] Local 500K dataset not found.")
    print("[DATA] Falling back to streaming RAID sampling.")
    df = collect_raid_sample(FALLBACK_SAMPLE_SIZE, SEED)

required_columns = {
    "id", "source_id", "model", "domain", "generation", "attack"
}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required RAID columns: {sorted(missing)}")

df["label"] = df["model"].apply(raid_label)
df["text"] = df["generation"].apply(clean_text)

before = len(df)
df = df[df["text"].str.len() >= 50].copy()
df = df.drop_duplicates("id").reset_index(drop=True)

print("[OK] Raw rows:", f"{before:,}")
print("[OK] Rows after cleaning:", f"{len(df):,}")
print("[OK] Duplicate IDs:", df["id"].duplicated().sum())
print("[OK] Human:", int((df["label"] == 0).sum()))
print("[OK] AI:", int((df["label"] == 1).sum()))

[DATA] Loading local 500K dataset: E:\TexTrace1\data\processed\raid_500k.csv
[OK] Raw rows: 500,000
[OK] Rows after cleaning: 499,661
[OK] Duplicate IDs: 0
[OK] Human: 160448
[OK] AI: 339213


## 3. EDA OUTPUTS

In [5]:
df["char_length"] = df["text"].str.len()
df["word_length"] = df["text"].str.split().str.len()

eda_summary = df[["char_length", "word_length"]].describe()
eda_summary.to_csv(REPORTS_DIR / "eda_text_length_summary.csv")

domain_counts = df["domain"].value_counts().sort_index()
domain_counts.to_csv(REPORTS_DIR / "domain_distribution.csv")

label_counts = (
    df["label"].map({0: "Human", 1: "AI"})
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)
label_counts.to_csv(REPORTS_DIR / "label_distribution.csv", index=False)

print("[OK] EDA summaries saved.")

[OK] EDA summaries saved.


## 4. GROUP-AWARE SPLIT

In [6]:
df["group_id"] = df["source_id"].fillna(df["id"]).astype(str)

splitter_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)
train_idx, temp_idx = next(
    splitter_1.split(df, y=df["label"], groups=df["group_id"])
)

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

splitter_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED
)
val_idx, test_idx = next(
    splitter_2.split(
        temp_df,
        y=temp_df["label"],
        groups=temp_df["group_id"]
    )
)

validation_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_groups = set(train_df["group_id"])
val_groups = set(validation_df["group_id"])
test_groups = set(test_df["group_id"])

print("\n" + "-" * 78)
print("GROUP-AWARE SPLIT")
print("-" * 78)
print("[OK] Train:", f"{len(train_df):,}")
print("[OK] Validation:", f"{len(validation_df):,}")
print("[OK] Test:", f"{len(test_df):,}")
print("[OK] Train ∩ Validation:", len(train_groups & val_groups))
print("[OK] Train ∩ Test:", len(train_groups & test_groups))
print("[OK] Validation ∩ Test:", len(val_groups & test_groups))

if train_groups & val_groups or train_groups & test_groups or val_groups & test_groups:
    raise RuntimeError("Group leakage detected.")

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
validation_df.to_csv(PROCESSED_DIR / "validation.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("[OK] Split files saved.")


------------------------------------------------------------------------------
GROUP-AWARE SPLIT
------------------------------------------------------------------------------
[OK] Train: 349,836
[OK] Validation: 75,012
[OK] Test: 74,813
[OK] Train ∩ Validation: 0
[OK] Train ∩ Test: 0
[OK] Validation ∩ Test: 0
[OK] Split files saved.


## 5. EVALUATION FUNCTIONS

In [7]:
X_train = train_df["text"].tolist()
y_train = train_df["label"].to_numpy()

X_val = validation_df["text"].tolist()
y_val = validation_df["label"].to_numpy()

X_test = test_df["text"].tolist()
y_test = test_df["label"].to_numpy()


def evaluate_scores(y_true, y_score, name, threshold=0.5):
    """Return standard metrics from a continuous AI score."""
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    result = {
        "Model": name,
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

    print(f"\n[{name}]")
    print(
        f"Accuracy={result['Accuracy']:.4f} | "
        f"Precision={result['Precision']:.4f} | "
        f"Recall={result['Recall']:.4f} | "
        f"F1={result['F1']:.4f} | "
        f"ROC-AUC={result['ROC-AUC']:.4f} | "
        f"FPR={result['FPR']:.4f}"
    )
    print("Confusion matrix:")
    print(np.array([[tn, fp], [fn, tp]]))
    return result


def print_step(message):
    print("\n" + "=" * 78)
    print(message)
    print("=" * 78)

print("[OK] Section completed: 5. EVALUATION FUNCTIONS")

[OK] Section completed: 5. EVALUATION FUNCTIONS


## 6. MODEL 1 — LOGISTIC REGRESSION

In [8]:
print_step("MODEL 1 — TF-IDF + LOGISTIC REGRESSION")

lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=250_000,
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

t0 = time.time()
lr_model.fit(X_train, y_train)
lr_train_seconds = time.time() - t0

lr_val_score = lr_model.predict_proba(X_val)[:, 1]
lr_test_score = lr_model.predict_proba(X_test)[:, 1]

lr_val_metrics = evaluate_scores(
    y_val, lr_val_score, "TF-IDF + Logistic Regression — Validation"
)
lr_test_metrics = evaluate_scores(
    y_test, lr_test_score, "TF-IDF + Logistic Regression — Test"
)

print("[OK] Logistic Regression complete.")
print(f"[OK] Training time: {lr_train_seconds:.2f} seconds.")


MODEL 1 — TF-IDF + LOGISTIC REGRESSION

[TF-IDF + Logistic Regression — Validation]
Accuracy=0.8607 | Precision=0.8909 | Recall=0.9059 | F1=0.8983 | ROC-AUC=0.9329 | FPR=0.2349
Confusion matrix:
[[18417  5654]
 [ 4794 46147]]

[TF-IDF + Logistic Regression — Test]
Accuracy=0.8624 | Precision=0.8915 | Recall=0.9076 | F1=0.8995 | ROC-AUC=0.9346 | FPR=0.2329
Confusion matrix:
[[18466  5606]
 [ 4687 46054]]
[OK] Logistic Regression complete.
[OK] Training time: 376.10 seconds.


## 7. MODEL 2 — CALIBRATED LINEAR SVM

In [9]:
print_step("MODEL 2 — TF-IDF + CALIBRATED LINEAR SVM")

svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=250_000,
    )),
    ("classifier", CalibratedClassifierCV(
        LinearSVC(
            class_weight="balanced",
            random_state=SEED,
        ),
        cv=3,
    )),
])

t0 = time.time()
svm_model.fit(X_train, y_train)
svm_train_seconds = time.time() - t0

svm_val_score = svm_model.predict_proba(X_val)[:, 1]
svm_test_score = svm_model.predict_proba(X_test)[:, 1]

svm_val_metrics = evaluate_scores(
    y_val, svm_val_score, "TF-IDF + Calibrated Linear SVM — Validation"
)
svm_test_metrics = evaluate_scores(
    y_test, svm_test_score, "TF-IDF + Calibrated Linear SVM — Test"
)

print("[OK] SVM complete.")
print(f"[OK] Training time: {svm_train_seconds:.2f} seconds.")


MODEL 2 — TF-IDF + CALIBRATED LINEAR SVM

[TF-IDF + Calibrated Linear SVM — Validation]
Accuracy=0.8333 | Precision=0.8199 | Recall=0.9668 | F1=0.8873 | ROC-AUC=0.9230 | FPR=0.4494
Confusion matrix:
[[13254 10817]
 [ 1690 49251]]

[TF-IDF + Calibrated Linear SVM — Test]
Accuracy=0.8348 | Precision=0.8210 | Recall=0.9673 | F1=0.8882 | ROC-AUC=0.9249 | FPR=0.4445
Confusion matrix:
[[13372 10700]
 [ 1659 49082]]
[OK] SVM complete.
[OK] Training time: 362.42 seconds.


## 8. MODEL 3 — XGBOOST + LINGUISTIC FEATURES

In [10]:
print_step("MODEL 3 — XGBOOST + LINGUISTIC FEATURES")


def linguistic_features(text):
    text = text if isinstance(text, str) else ""
    words = re.findall(r"\b\w+\b", text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]

    n_words = len(words)
    n_sent = max(1, len(sentences))
    n_chars = len(text)
    unique = len(set(w.lower() for w in words))

    return {
        "char_count": n_chars,
        "word_count": n_words,
        "sentence_count": n_sent,
        "avg_word_length": (
            np.mean([len(w) for w in words]) if words else 0
        ),
        "avg_sentence_length": n_words / n_sent,
        "unique_word_ratio": unique / n_words if n_words else 0,
        "digit_ratio": (
            sum(c.isdigit() for c in text) / n_chars
            if n_chars else 0
        ),
        "uppercase_ratio": (
            sum(c.isupper() for c in text) / n_chars
            if n_chars else 0
        ),
        "punctuation_ratio": (
            sum(
                not c.isalnum() and not c.isspace()
                for c in text
            ) / n_chars
            if n_chars else 0
        ),
        "question_count": text.count("?"),
        "exclamation_count": text.count("!"),
        "comma_count": text.count(","),
        "semicolon_count": text.count(";"),
        "colon_count": text.count(":"),
        "newline_count": text.count("\n"),
    }


def make_linguistic_features(texts):
    return pd.DataFrame(
        [linguistic_features(text) for text in texts]
    )


X_train_ling = make_linguistic_features(X_train)
X_val_ling = make_linguistic_features(X_val)
X_test_ling = make_linguistic_features(X_test)

xgb_model = None
xgb_val_score = None
xgb_test_score = None
xgb_val_metrics = None
xgb_test_metrics = None

if XGB_AVAILABLE:
    xgb_model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
    )

    t0 = time.time()
    xgb_model.fit(
        X_train_ling,
        y_train,
        eval_set=[(X_val_ling, y_val)],
        verbose=False,
    )
    xgb_train_seconds = time.time() - t0

    xgb_val_score = xgb_model.predict_proba(
        X_val_ling
    )[:, 1]
    xgb_test_score = xgb_model.predict_proba(
        X_test_ling
    )[:, 1]

    xgb_val_metrics = evaluate_scores(
        y_val,
        xgb_val_score,
        "XGBoost + Linguistic Features — Validation",
    )
    xgb_test_metrics = evaluate_scores(
        y_test,
        xgb_test_score,
        "XGBoost + Linguistic Features — Test",
    )

    print("[OK] XGBoost complete.")
    print(f"[OK] Training time: {xgb_train_seconds:.2f} seconds.")
else:
    print("[SKIP] XGBoost is unavailable.")


MODEL 3 — XGBOOST + LINGUISTIC FEATURES

[XGBoost + Linguistic Features — Validation]
Accuracy=0.8539 | Precision=0.8723 | Recall=0.9196 | F1=0.8953 | ROC-AUC=0.9229 | FPR=0.2850
Confusion matrix:
[[17210  6861]
 [ 4095 46846]]

[XGBoost + Linguistic Features — Test]
Accuracy=0.8551 | Precision=0.8730 | Recall=0.9203 | F1=0.8960 | ROC-AUC=0.9266 | FPR=0.2823
Confusion matrix:
[[17276  6796]
 [ 4042 46699]]
[OK] XGBoost complete.
[OK] Training time: 3.17 seconds.


## 9. MODEL 4 — DISTILROBERTA

In [11]:
# ============================================================
# FIX — MISSING SKLEARN METRIC IMPORT
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
)

print("[OK] precision_recall_fscore_support imported.")
print("[OK] Transformer evaluation metrics are ready.")

[OK] precision_recall_fscore_support imported.
[OK] Transformer evaluation metrics are ready.


In [12]:
# ============================================================
# DISTILROBERTA
# Transformer-Based AI vs Human Text Classification
# ============================================================

print("=" * 78)
print("DISTILROBERTA — TRANSFORMER MODEL")
print("=" * 78)

# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

import time
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
)

print("[OK] Transformers imported.")
print("[OK] Hugging Face Dataset imported.")
print("[OK] Evaluation metrics imported.")


# ------------------------------------------------------------
# 2. Check GPU
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("DEVICE CHECK")
print("-" * 78)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[OK] Device: {device}")

if torch.cuda.is_available():
    print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"[OK] GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("[WARNING] CUDA is not available.")


# ------------------------------------------------------------
# 3. Check dataset
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("DATASET CHECK")
print("-" * 78)

print(f"Train dataframe shape: {train_df.shape}")
print(f"Validation dataframe shape: {validation_df.shape}")

# Make sure required columns exist
required_columns = ["text", "label"]

for column in required_columns:
    if column not in train_df.columns:
        raise KeyError(f"Required column '{column}' not found in train_df.")

    if column not in validation_df.columns:
        raise KeyError(
            f"Required column '{column}' not found in validation_df."
        )

print("[OK] Required columns found.")


# ------------------------------------------------------------
# 4. Prepare balanced 50K training subset
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("CREATING DISTILROBERTA TRAINING SUBSET")
print("-" * 78)

# Make sure labels are numeric
train_df["label"] = train_df["label"].astype(int)
validation_df["label"] = validation_df["label"].astype(int)

# Human = 0
# AI = 1

human_train = train_df[
    train_df["label"] == 0
].sample(
    n=25_000,
    random_state=SEED,
)

ai_train = train_df[
    train_df["label"] == 1
].sample(
    n=25_000,
    random_state=SEED,
)

transformer_train_df = (
    __import__("pandas")
    .concat(
        [human_train, ai_train],
        ignore_index=True,
    )
    .sample(
        frac=1,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

print(f"[OK] Training subset size: {len(transformer_train_df):,}")
print(
    f"[OK] Human samples: "
    f"{(transformer_train_df['label'] == 0).sum():,}"
)
print(
    f"[OK] AI samples: "
    f"{(transformer_train_df['label'] == 1).sum():,}"
)


# ------------------------------------------------------------
# 5. Create Hugging Face datasets
# ------------------------------------------------------------

train_hf = Dataset.from_pandas(
    transformer_train_df[["text", "label"]],
    preserve_index=False,
)

val_hf = Dataset.from_pandas(
    validation_df[["text", "label"]],
    preserve_index=False,
)

print("[OK] Hugging Face training dataset created.")
print("[OK] Hugging Face validation dataset created.")


# ------------------------------------------------------------
# 6. Load DistilRoBERTa tokenizer
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("LOADING DISTILROBERTA TOKENIZER")
print("-" * 78)

MODEL_NAME = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(f"[OK] Tokenizer loaded: {MODEL_NAME}")


# ------------------------------------------------------------
# 7. Tokenization
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("TOKENIZING TEXT")
print("-" * 78)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
    )


train_hf = train_hf.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing training data",
)

val_hf = val_hf.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing validation data",
)

print("[OK] Training data tokenized.")
print("[OK] Validation data tokenized.")


# ------------------------------------------------------------
# 8. Dynamic padding
# ------------------------------------------------------------

# Important:
# Do NOT use padding="max_length" here.
# Dynamic padding prevents:
# "expected sequence of length 512 at dim 1"

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
)

print("[OK] Dynamic padding enabled.")


# ------------------------------------------------------------
# 9. Load DistilRoBERTa classification model
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("LOADING DISTILROBERTA MODEL")
print("-" * 78)

transformer_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "Human",
        1: "AI",
    },
    label2id={
        "Human": 0,
        "AI": 1,
    },
)

print("[OK] DistilRoBERTa model loaded.")


# ------------------------------------------------------------
# 10. Enable gradient checkpointing
# ------------------------------------------------------------

if torch.cuda.is_available():
    transformer_model.gradient_checkpointing_enable()
    print("[OK] Gradient checkpointing enabled.")


# ------------------------------------------------------------
# 11. Evaluation metrics
# ------------------------------------------------------------

def hf_metrics(eval_pred):

    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    # Some Transformers versions may return a tuple
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(
        logits,
        axis=-1,
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            pos_label=1,
            zero_division=0,
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions,
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


print("[OK] Evaluation metrics configured.")


# ------------------------------------------------------------
# 12. Training configuration
# ------------------------------------------------------------

CHECKPOINT_DIR = (
    MODELS_DIR / "distilroberta_checkpoints"
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=4,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=SEED,

    save_total_limit=2,
)

print("[OK] TrainingArguments configured.")


# ------------------------------------------------------------
# 13. Create Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=transformer_model,
    args=training_args,

    train_dataset=train_hf,
    eval_dataset=val_hf,

    data_collator=data_collator,

    compute_metrics=hf_metrics,
)

print("[OK] Hugging Face Trainer created.")


# ------------------------------------------------------------
# 14. Train DistilRoBERTa
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("STARTING DISTILROBERTA TRAINING")
print("=" * 78)

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time

print("\n" + "=" * 78)
print("DISTILROBERTA TRAINING COMPLETED")
print("=" * 78)

print(
    f"[OK] Training time: "
    f"{training_time / 60:.2f} minutes"
)

if hasattr(train_result, "training_loss"):
    print(
        f"[OK] Final training loss: "
        f"{train_result.training_loss:.4f}"
    )


# ------------------------------------------------------------
# 15. Validation evaluation
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("DISTILROBERTA VALIDATION")
print("-" * 78)

eval_results = trainer.evaluate()

for key, value in eval_results.items():

    if isinstance(value, (int, float, np.integer, np.floating)):
        print(
            f"{key}: {float(value):.4f}"
        )


# ------------------------------------------------------------
# 16. Generate validation probabilities
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("GENERATING VALIDATION PREDICTIONS")
print("-" * 78)

prediction_output = trainer.predict(
    val_hf
)

val_logits = prediction_output.predictions

if isinstance(val_logits, tuple):
    val_logits = val_logits[0]

# Softmax
val_exp = np.exp(
    val_logits - np.max(
        val_logits,
        axis=1,
        keepdims=True,
    )
)

val_probabilities = (
    val_exp /
    val_exp.sum(
        axis=1,
        keepdims=True,
    )
)

# Probability of AI class
transformer_val_score = val_probabilities[:, 1]

print(
    f"[OK] Validation predictions: "
    f"{len(transformer_val_score):,}"
)

print(
    f"[OK] AI probability range: "
    f"{transformer_val_score.min():.4f} "
    f"to "
    f"{transformer_val_score.max():.4f}"
)


# ------------------------------------------------------------
# 17. Evaluate using project's standard evaluation function
# ------------------------------------------------------------

if "evaluate_scores" in globals():

    transformer_results = evaluate_scores(
        y_val,
        transformer_val_score,
        "DistilRoBERTa — Validation",
    )

    print("\n[OK] Project evaluation completed.")

else:

    # Fallback evaluation if evaluate_scores()
    # is not available in the notebook.

    transformer_pred = (
        transformer_val_score >= 0.50
    ).astype(int)

    transformer_results = {
        "Accuracy": accuracy_score(
            y_val,
            transformer_pred,
        ),
        "Precision": precision_score(
            y_val,
            transformer_pred,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_val,
            transformer_pred,
            zero_division=0,
        ),
        "F1": f1_score(
            y_val,
            transformer_pred,
            zero_division=0,
        ),
    }

    print("\n[OK] Fallback evaluation completed.")

    for key, value in transformer_results.items():
        print(
            f"{key}: {value:.4f}"
        )


# ------------------------------------------------------------
# 18. Save DistilRoBERTa model
# ------------------------------------------------------------

FINAL_TRANSFORMER_DIR = (
    MODELS_DIR / "distilroberta"
)

FINAL_TRANSFORMER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

trainer.save_model(
    str(FINAL_TRANSFORMER_DIR)
)

tokenizer.save_pretrained(
    str(FINAL_TRANSFORMER_DIR)
)

print("\n" + "-" * 78)
print("MODEL SAVING")
print("-" * 78)

print(
    f"[OK] DistilRoBERTa model saved to:\n"
    f"{FINAL_TRANSFORMER_DIR}"
)

print("\n" + "=" * 78)
print("DISTILROBERTA CELL COMPLETED SUCCESSFULLY")
print("=" * 78)

DISTILROBERTA — TRANSFORMER MODEL
[OK] Transformers imported.
[OK] Hugging Face Dataset imported.
[OK] Evaluation metrics imported.

------------------------------------------------------------------------------
DEVICE CHECK
------------------------------------------------------------------------------
[OK] Device: cuda
[OK] GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[OK] GPU Memory: 8.00 GB

------------------------------------------------------------------------------
DATASET CHECK
------------------------------------------------------------------------------
Train dataframe shape: (349836, 16)
Validation dataframe shape: (75012, 16)
[OK] Required columns found.

------------------------------------------------------------------------------
CREATING DISTILROBERTA TRAINING SUBSET
------------------------------------------------------------------------------
[OK] Training subset size: 50,000
[OK] Human samples: 25,000
[OK] AI samples: 25,000
[OK] Hugging Face training dataset created.
[OK

Tokenizing validation data: 100%|██████████| 75012/75012 [00:15<00:00, 4782.42 examples/s]


[OK] Training data tokenized.
[OK] Validation data tokenized.
[OK] Dynamic padding enabled.

------------------------------------------------------------------------------
LOADING DISTILROBERTA MODEL
------------------------------------------------------------------------------


Loading weights: 100%|██████████| 101/101 [00:00<00:00, 704.91it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[OK] DistilRoBERTa model loaded.
[OK] Gradient checkpointing enabled.
[OK] Evaluation metrics configured.
[OK] TrainingArguments configured.
[OK] Hugging Face Trainer created.

STARTING DISTILROBERTA TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.735267,0.540940,0.836400,0.835978,0.944387,0.886881
2,0.615277,0.442586,0.893670,0.895706,0.954575,0.924204


... (3 outputs left out)

eval_loss: 0.4426
eval_accuracy: 0.8937
eval_precision: 0.8957
eval_recall: 0.9546
eval_f1: 0.9242

------------------------------------------------------------------------------
GENERATING VALIDATION PREDICTIONS
------------------------------------------------------------------------------


[OK] Validation predictions: 75,012
[OK] AI probability range: 0.0002 to 0.9999

[DistilRoBERTa — Validation]
Accuracy=0.8937 | Precision=0.8957 | Recall=0.9546 | F1=0.9242 | ROC-AUC=0.9644 | FPR=0.2352
Confusion matrix:
[[18409  5662]
 [ 2314 48627]]

[OK] Project evaluation completed.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.76it/s]


------------------------------------------------------------------------------
MODEL SAVING
------------------------------------------------------------------------------
[OK] DistilRoBERTa model saved to:
E:\TexTrace1\models\distilroberta

DISTILROBERTA CELL COMPLETED SUCCESSFULLY


## 10. MODEL COMPARISON

In [16]:

print_step("VALIDATION MODEL COMPARISON")

# ------------------------------------------------------------
# 1. Start with Logistic Regression and calibrated Linear SVM
# ------------------------------------------------------------

validation_metrics = [
    lr_val_metrics,
    svm_val_metrics,
]

score_map = {
    "LR": lr_val_score,
    "SVM": svm_val_score,
}


# ------------------------------------------------------------
# 2. Add XGBoost if it was successfully trained
# ------------------------------------------------------------

if (
    globals().get("xgb_val_metrics") is not None
    and globals().get("xgb_val_score") is not None
):
    validation_metrics.append(xgb_val_metrics)
    score_map["XGBoost"] = xgb_val_score

else:
    print("[SKIP] XGBoost validation results are unavailable.")


# ------------------------------------------------------------
# 3. Add DistilRoBERTa if its validation cell has run
#    Correct variable name: transformer_results
# ------------------------------------------------------------

transformer_metrics = globals().get("transformer_results")
transformer_score = globals().get("transformer_val_score")

if (
    transformer_metrics is not None
    and transformer_score is not None
):
    # Ensure predictions correspond to the same validation labels.
    if len(transformer_score) != len(y_val):
        raise ValueError(
            "DistilRoBERTa validation predictions do not match "
            f"y_val: {len(transformer_score)} predictions versus "
            f"{len(y_val)} labels. Check the validation split "
            "before comparing or combining these models."
        )

    validation_metrics.append(transformer_metrics)
    score_map["DistilRoBERTa"] = transformer_score

else:
    print(
        "[SKIP] DistilRoBERTa validation results are unavailable. "
        "Run its training and validation cell first if you want "
        "to include it in the comparison."
    )


# ------------------------------------------------------------
# 4. Check that every included model has the same label count
# ------------------------------------------------------------

for model_name, scores in score_map.items():
    if len(scores) != len(y_val):
        raise ValueError(
            f"{model_name} has {len(scores)} validation predictions, "
            f"but y_val has {len(y_val)} labels."
        )


# ------------------------------------------------------------
# 5. Create and save the comparison table
# ------------------------------------------------------------

validation_results = pd.DataFrame(validation_metrics)

if "F1" in validation_results.columns:
    validation_results = validation_results.sort_values(
        "F1",
        ascending=False,
    )

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

validation_results.to_csv(
    REPORTS_DIR / "validation_model_results.csv",
    index=False,
)

print("\nVALIDATION MODEL COMPARISON")
print("=" * 78)
print(validation_results.to_string(index=False))

print("\nModels available for ensemble search:")
for model_name in score_map:
    print(f"  - {model_name}")

print("\n[OK] Section completed: 10. MODEL COMPARISON")



VALIDATION MODEL COMPARISON

VALIDATION MODEL COMPARISON
                                      Model  Threshold  Accuracy  Precision   Recall       F1  ROC-AUC      FPR    TN    FP   FN    TP
                 DistilRoBERTa — Validation        0.5  0.893670   0.895706 0.954575 0.924204 0.964378 0.235221 18409  5662 2314 48627
  TF-IDF + Logistic Regression — Validation        0.5  0.860716   0.890852 0.905891 0.898308 0.932899 0.234888 18417  5654 4794 46147
 XGBoost + Linguistic Features — Validation        0.5  0.853943   0.872251 0.919613 0.895306 0.922869 0.285032 17210  6861 4095 46846
TF-IDF + Calibrated Linear SVM — Validation        0.5  0.833267   0.819921 0.966824 0.887333 0.922992 0.449379 13254 10817 1690 49251

Models available for ensemble search:
  - LR
  - SVM
  - XGBoost
  - DistilRoBERTa

[OK] Section completed: 10. MODEL COMPARISON


## 11. VALIDATION-ONLY ENSEMBLE WEIGHT + THRESHOLD SEARCH

In [14]:
print_step("VALIDATION-ONLY ENSEMBLE SEARCH")

model_names = list(score_map.keys())
score_matrix = np.column_stack(
    [score_map[name] for name in model_names]
)

# Deterministic candidate set.
rng = np.random.default_rng(SEED)

candidate_weights = [
    np.eye(len(model_names))[i]
    for i in range(len(model_names))
]

candidate_weights += [
    rng.dirichlet(
        np.ones(len(model_names)
    ))
    for _ in range(5000)
]

thresholds = np.round(
    np.arange(0.40, 0.701, 0.01),
    2,
)

best = {
    "F1": -1,
    "Accuracy": -1,
    "weights": None,
    "threshold": None,
    "score": None,
}

for weights in candidate_weights:
    combined = score_matrix @ weights

    for threshold in thresholds:
        pred = (combined >= threshold).astype(int)
        f1 = f1_score(
            y_val,
            pred,
            zero_division=0,
        )

        if f1 > best["F1"]:
            best["F1"] = float(f1)
            best["weights"] = weights.copy()
            best["threshold"] = float(threshold)
            best["score"] = combined.copy()

best_weights = best["weights"]
best_threshold = best["threshold"]
ensemble_val_score = best["score"]

ensemble_val_metrics = evaluate_scores(
    y_val,
    ensemble_val_score,
    "TextTrace Ensemble — Validation",
    threshold=best_threshold,
)

print("[OK] Validation-only ensemble search complete.")
print("[OK] Selected threshold:", best_threshold)
for name, weight in zip(model_names, best_weights):
    print(f"[OK] {name:18s}: {weight:.4f}")


VALIDATION-ONLY ENSEMBLE SEARCH

[TextTrace Ensemble — Validation]
Accuracy=0.9340 | Precision=0.9422 | Recall=0.9617 | F1=0.9519 | ROC-AUC=0.9790 | FPR=0.1248
Confusion matrix:
[[21066  3005]
 [ 1949 48992]]
[OK] Validation-only ensemble search complete.
[OK] Selected threshold: 0.56
[OK] LR                : 0.3825
[OK] SVM               : 0.0034
[OK] XGBoost           : 0.3631
[OK] DistilRoBERTa     : 0.2509


## 12. FINAL HELD-OUT TEST

In [17]:
print_step("FINAL HELD-OUT TEST")

from scipy.special import softmax

test_score_map = {
    "LR": lr_test_score,
    "SVM": svm_test_score,
}

if xgb_test_score is not None:
    test_score_map["XGBoost"] = xgb_test_score

# Evaluate DistilRoBERTa only if it was included
# in the validation ensemble.
if "DistilRoBERTa" in model_names:

    if "trainer" not in globals():
        raise RuntimeError(
            "DistilRoBERTa is in the validation ensemble, "
            "but its trained Trainer is unavailable. "
            "Run the DistilRoBERTa training cell first."
        )

    test_hf = Dataset.from_pandas(
        test_df[["text", "label"]],
        preserve_index=False,
    )

    test_hf = test_hf.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
    )

    test_output = trainer.predict(test_hf)

    test_logits = test_output.predictions

    if isinstance(test_logits, tuple):
        test_logits = test_logits[0]

    transformer_test_score = softmax(
        test_logits,
        axis=1,
    )[:, 1]

    transformer_test_metrics = evaluate_scores(
        y_test,
        transformer_test_score,
        "DistilRoBERTa — Test",
    )

    test_score_map["DistilRoBERTa"] = (
        transformer_test_score
    )

# Ensure all ensemble models have test scores.
missing_models = [
    name for name in model_names
    if name not in test_score_map
]

if missing_models:
    raise ValueError(
        f"Missing test scores for: {missing_models}"
    )

test_matrix = np.column_stack(
    [test_score_map[name] for name in model_names]
)

ensemble_test_score = test_matrix @ best_weights

ensemble_test_metrics = evaluate_scores(
    y_test,
    ensemble_test_score,
    "TextTrace Ensemble — Final Test",
    threshold=best_threshold,
)

print("[OK] Final held-out test completed.")
print("[IMPORTANT] Test set was not used to choose ensemble weights.")


FINAL HELD-OUT TEST


Map: 100%|██████████| 74813/74813 [00:18<00:00, 4055.65 examples/s]



[DistilRoBERTa — Test]
Accuracy=0.8919 | Precision=0.8931 | Recall=0.9549 | F1=0.9230 | ROC-AUC=0.9647 | FPR=0.2410
Confusion matrix:
[[18271  5801]
 [ 2287 48454]]

[TextTrace Ensemble — Final Test]
Accuracy=0.9339 | Precision=0.9420 | Recall=0.9618 | F1=0.9518 | ROC-AUC=0.9805 | FPR=0.1248
Confusion matrix:
[[21067  3005]
 [ 1938 48803]]
[OK] Final held-out test completed.
[IMPORTANT] Test set was not used to choose ensemble weights.


## 13. FINAL MODEL RESULTS TABLE

In [18]:
final_metrics = [
    lr_test_metrics,
    svm_test_metrics,
]

if xgb_test_metrics is not None:
    final_metrics.append(xgb_test_metrics)

if transformer_test_metrics is not None:
    final_metrics.append(transformer_test_metrics)

final_metrics.append(ensemble_test_metrics)

final_results_df = (
    pd.DataFrame(final_metrics)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

final_results_df.to_csv(
    REPORTS_DIR / "final_model_results.csv",
    index=False,
)

print_step("FINAL TEST RESULTS")
print(final_results_df.to_string(index=False))

print("[OK] Section completed: 13. FINAL MODEL RESULTS TABLE")


FINAL TEST RESULTS
                                Model  Threshold  Accuracy  Precision   Recall       F1  ROC-AUC      FPR    TN    FP   FN    TP
      TextTrace Ensemble — Final Test       0.56  0.933929   0.941997 0.961806 0.951799 0.980516 0.124834 21067  3005 1938 48803
                 DistilRoBERTa — Test       0.50  0.891890   0.893079 0.954928 0.922968 0.964749 0.240985 18271  5801 2287 48454
  TF-IDF + Logistic Regression — Test       0.50  0.862417   0.891483 0.907629 0.899483 0.934565 0.232885 18466  5606 4687 46054
 XGBoost + Linguistic Features — Test       0.50  0.855132   0.872960 0.920341 0.896024 0.926636 0.282320 17276  6796 4042 46699
TF-IDF + Calibrated Linear SVM — Test       0.50  0.834801   0.821016 0.967305 0.888177 0.924939 0.444500 13372 10700 1659 49082
[OK] Section completed: 13. FINAL MODEL RESULTS TABLE


## 14. CONFUSION MATRIX + ROC CURVES

In [19]:
def save_confusion_matrix(y_true, score, threshold, filename, title):
    pred = (score >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])

    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111)
    ax.imshow(cm)

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
            )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Human", "AI"])
    ax.set_yticklabels(["Human", "AI"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / filename, dpi=220)
    plt.close(fig)


save_confusion_matrix(
    y_test,
    ensemble_test_score,
    best_threshold,
    "01_final_ensemble_confusion_matrix.png",
    "TextTrace Final Ensemble — Confusion Matrix",
)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

for name in model_names:
    fpr, tpr, _ = roc_curve(
        y_test,
        test_score_map[name],
    )
    auc = roc_auc_score(
        y_test,
        test_score_map[name],
    )
    ax.plot(
        fpr,
        tpr,
        label=f"{name} (AUC={auc:.3f})",
    )

fpr, tpr, _ = roc_curve(
    y_test,
    ensemble_test_score,
)
auc = roc_auc_score(
    y_test,
    ensemble_test_score,
)
ax.plot(
    fpr,
    tpr,
    label=f"Ensemble (AUC={auc:.3f})",
    linewidth=2.5,
)

ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("TextTrace ROC Curves — Held-Out Test")
ax.legend()
fig.tight_layout()
fig.savefig(
    CHARTS_DIR / "02_roc_curves_test.png",
    dpi=220,
)
plt.close(fig)

# F1 comparison.
plot_df = final_results_df.copy()
fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111)
ax.bar(plot_df["Model"], plot_df["F1"])
ax.set_ylabel("F1")
ax.set_title("TextTrace Model Comparison — F1")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(
    CHARTS_DIR / "03_model_f1_comparison.png",
    dpi=220,
)
plt.close(fig)

# Accuracy/precision/recall.
metric_cols = ["Accuracy", "Precision", "Recall", "F1"]
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)
x = np.arange(len(plot_df))
width = 0.18

for i, metric in enumerate(metric_cols):
    ax.bar(
        x + (i - 1.5) * width,
        plot_df[metric],
        width,
        label=metric,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["Model"],
    rotation=35,
    ha="right",
)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("TextTrace Test Metrics")
ax.legend()
fig.tight_layout()
fig.savefig(
    CHARTS_DIR / "04_test_metrics_comparison.png",
    dpi=220,
)
plt.close(fig)

print("[OK] Core evaluation charts saved.")

[OK] Core evaluation charts saved.


## 15. XAI — LOGISTIC REGRESSION

In [20]:
def explain_lr_text(text, pipeline, top_n=15):
    vec = pipeline.named_steps["tfidf"]
    clf = pipeline.named_steps["classifier"]

    x = vec.transform([text])
    names = np.array(vec.get_feature_names_out())
    idx = x.nonzero()[1]

    contributions = (
        x[0, idx].toarray().ravel()
        * clf.coef_[0, idx]
    )

    order = np.argsort(
        np.abs(contributions)
    )[::-1][:top_n]

    return pd.DataFrame({
        "feature": names[idx][order],
        "contribution": contributions[order],
    })


xai_df = explain_lr_text(
    test_df.iloc[0]["text"],
    lr_model,
    top_n=15,
)

xai_df.to_csv(
    REPORTS_DIR / "lr_xai_example.csv",
    index=False,
)

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111)
xai_plot = xai_df.sort_values("contribution")
ax.barh(
    xai_plot["feature"],
    xai_plot["contribution"],
)
ax.axvline(0, linestyle="--")
ax.set_xlabel("Contribution to AI direction (+) / Human direction (-)")
ax.set_title("Example Logistic Regression Explanation")
fig.tight_layout()
fig.savefig(
    CHARTS_DIR / "05_lr_xai_contributions.png",
    dpi=220,
)
plt.close(fig)

print("[OK] Logistic Regression XAI output saved.")

[OK] Logistic Regression XAI output saved.


## 16. XGBOOST FEATURE IMPORTANCE

In [21]:
if xgb_model is not None:
    xgb_importance = pd.DataFrame({
        "feature": X_train_ling.columns,
        "importance": xgb_model.feature_importances_,
    }).sort_values(
        "importance",
        ascending=False,
    )

    xgb_importance.to_csv(
        REPORTS_DIR / "xgb_feature_importance.csv",
        index=False,
    )

    top_xgb = xgb_importance.head(15).sort_values(
        "importance"
    )

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111)
    ax.barh(
        top_xgb["feature"],
        top_xgb["importance"],
    )
    ax.set_xlabel("Feature importance")
    ax.set_title("Top XGBoost Linguistic Features")
    fig.tight_layout()
    fig.savefig(
        CHARTS_DIR / "06_xgb_feature_importance.png",
        dpi=220,
    )
    plt.close(fig)

    print("[OK] XGBoost feature importance saved.")

[OK] XGBoost feature importance saved.


## 17. ERROR ANALYSIS

In [22]:
error_df = test_df[
    ["id", "domain", "model", "attack", "text", "label"]
].copy()

error_df["ai_score"] = ensemble_test_score
error_df["prediction"] = (
    ensemble_test_score >= best_threshold
).astype(int)

error_df["error_type"] = "Correct"

error_df.loc[
    (error_df["label"] == 0)
    & (error_df["prediction"] == 1),
    "error_type",
] = "False Positive"

error_df.loc[
    (error_df["label"] == 1)
    & (error_df["prediction"] == 0),
    "error_type",
] = "False Negative"

errors_only = error_df[
    error_df["error_type"] != "Correct"
].copy()

errors_only.to_csv(
    REPORTS_DIR / "error_analysis.csv",
    index=False,
)

fp_by_domain = (
    errors_only[
        errors_only["error_type"] == "False Positive"
    ]["domain"]
    .value_counts()
    .sort_values(ascending=True)
)

fn_by_domain = (
    errors_only[
        errors_only["error_type"] == "False Negative"
    ]["domain"]
    .value_counts()
    .sort_values(ascending=True)
)

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111)
domains = sorted(
    set(fp_by_domain.index) | set(fn_by_domain.index)
)
fp_values = [
    fp_by_domain.get(d, 0)
    for d in domains
]
fn_values = [
    fn_by_domain.get(d, 0)
    for d in domains
]

x = np.arange(len(domains))
width = 0.38
ax.bar(x - width / 2, fp_values, width, label="False Positive")
ax.bar(x + width / 2, fn_values, width, label="False Negative")
ax.set_xticks(x)
ax.set_xticklabels(domains, rotation=35, ha="right")
ax.set_ylabel("Number of errors")
ax.set_title("TextTrace Error Analysis by Domain")
ax.legend()
fig.tight_layout()
fig.savefig(
    CHARTS_DIR / "07_error_analysis_by_domain.png",
    dpi=220,
)
plt.close(fig)

print("[OK] Error analysis saved.")
print("[OK] False positives:", int(
    (error_df["error_type"] == "False Positive").sum()
))
print("[OK] False negatives:", int(
    (error_df["error_type"] == "False Negative").sum()
))

[OK] Error analysis saved.
[OK] False positives: 3005
[OK] False negatives: 1938


## 18. ATTACK / ROBUSTNESS ANALYSIS

In [23]:
attack_df = error_df.copy()
attack_df["attack"] = attack_df["attack"].fillna("none")

attack_rows = []

for attack, group in attack_df.groupby("attack"):
    if len(group) < 2:
        continue

    attack_rows.append({
        "Attack": attack,
        "N": len(group),
        "Accuracy": accuracy_score(
            group["label"],
            group["prediction"],
        ),
        "Precision": precision_score(
            group["label"],
            group["prediction"],
            zero_division=0,
        ),
        "Recall": recall_score(
            group["label"],
            group["prediction"],
            zero_division=0,
        ),
        "F1": f1_score(
            group["label"],
            group["prediction"],
            zero_division=0,
        ),
    })

attack_results = pd.DataFrame(
    attack_rows
).sort_values(
    "F1",
    ascending=False,
)

attack_results.to_csv(
    REPORTS_DIR / "attack_results.csv",
    index=False,
)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)
attack_plot = attack_results.sort_values("F1")
ax.barh(
    attack_plot["Attack"],
    attack_plot["F1"],
)
ax.set_xlabel("F1")
ax.set_title("TextTrace Robustness — F1 by RAID Attack")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(
    CHARTS_DIR / "08_attack_robustness_f1.png",
    dpi=220,
)
plt.close(fig)

print("[OK] Attack robustness analysis saved.")

[OK] Attack robustness analysis saved.


## 19. ADAPTIVE / HARD-EXAMPLE EXPERIMENT

In [24]:
print_step("ADAPTIVE / HARD-EXAMPLE EXPERIMENT")

initial_train_score = lr_model.predict_proba(
    X_train
)[:, 1]

initial_train_pred = (
    initial_train_score >= 0.5
).astype(int)

hard_mask = (
    (initial_train_pred != y_train)
    | (np.abs(initial_train_score - 0.5) < 0.10)
)

sample_weight = np.ones(
    len(train_df),
    dtype=float,
)
sample_weight[hard_mask] = 2.0

adaptive_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=250_000,
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

adaptive_lr.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weight,
)

adaptive_val_score = adaptive_lr.predict_proba(
    X_val
)[:, 1]

adaptive_val_metrics = evaluate_scores(
    y_val,
    adaptive_val_score,
    "Adaptive Hard-Example Logistic Regression — Validation",
)

adaptive_comparison = pd.DataFrame([
    {
        "Stage": "Initial Logistic Regression",
        "F1": lr_val_metrics["F1"],
        "Accuracy": lr_val_metrics["Accuracy"],
        "Precision": lr_val_metrics["Precision"],
        "Recall": lr_val_metrics["Recall"],
        "ROC-AUC": lr_val_metrics["ROC-AUC"],
        "FPR": lr_val_metrics["FPR"],
    },
    {
        "Stage": "Hard-example weighted Logistic Regression",
        "F1": adaptive_val_metrics["F1"],
        "Accuracy": adaptive_val_metrics["Accuracy"],
        "Precision": adaptive_val_metrics["Precision"],
        "Recall": adaptive_val_metrics["Recall"],
        "ROC-AUC": adaptive_val_metrics["ROC-AUC"],
        "FPR": adaptive_val_metrics["FPR"],
    },
])

adaptive_comparison.to_csv(
    REPORTS_DIR / "adaptive_comparison.csv",
    index=False,
)

fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111)
ax.bar(
    adaptive_comparison["Stage"],
    adaptive_comparison["F1"],
)
ax.set_ylabel("Validation F1")
ax.set_title("Adaptive Hard-Example Experiment")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(
    CHARTS_DIR / "09_adaptive_learning_comparison.png",
    dpi=220,
)
plt.close(fig)

print(
    "[OK] Hard/uncertain training examples:",
    int(hard_mask.sum())
)

print("[OK] Section completed: 19. ADAPTIVE / HARD-EXAMPLE EXPERIMENT")


ADAPTIVE / HARD-EXAMPLE EXPERIMENT

[Adaptive Hard-Example Logistic Regression — Validation]
Accuracy=0.8584 | Precision=0.8703 | Recall=0.9302 | F1=0.8993 | ROC-AUC=0.9301 | FPR=0.2935
Confusion matrix:
[[17007  7064]
 [ 3554 47387]]
[OK] Hard/uncertain training examples: 58458
[OK] Section completed: 19. ADAPTIVE / HARD-EXAMPLE EXPERIMENT


## 20. STREAMLIT-READY DOCUMENT-LEVEL SCORING FUNCTION

In [25]:
def score_document_from_ensemble(
    text,
    lr_pipeline,
    svm_pipeline,
    xgb_classifier,
    transformer_trainer=None,
    transformer_tokenizer=None,
    weights=None,
    threshold=0.5,
):
    """
    Return the ML-stage result that Streamlit can later display.

    IMPORTANT:
    - ai_score is an AI-likelihood model score, not proof of authorship.
    - similarity_percentage is NOT produced here.
      It belongs to the separate similarity engine in Streamlit.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Document text is empty.")

    scores = []

    lr_score = lr_pipeline.predict_proba([text])[0, 1]
    svm_score = svm_pipeline.predict_proba([text])[0, 1]

    scores.extend([
        ("LR", float(lr_score)),
        ("SVM", float(svm_score)),
    ])

    if xgb_classifier is not None:
        xgb_features = make_linguistic_features([text])
        xgb_score = xgb_classifier.predict_proba(
            xgb_features
        )[0, 1]
        scores.append(("XGBoost", float(xgb_score)))

    if (
        transformer_trainer is not None
        and transformer_tokenizer is not None
    ):
        from datasets import Dataset
        from scipy.special import softmax

        one = Dataset.from_dict({
            "text": [text],
            "label": [0],
        })

        one = one.map(
            lambda batch: transformer_tokenizer(
                batch["text"],
                truncation=True,
                max_length=MAX_LENGTH,
            ),
            batched=True,
            remove_columns=["text"],
        )

        output = transformer_trainer.predict(one)
        score = softmax(
            output.predictions,
            axis=1,
        )[0, 1]

        scores.append(
            ("DistilRoBERTa", float(score))
        )

    score_dict = dict(scores)

    if weights is None:
        weights = {
            name: 1 / len(scores)
            for name, _ in scores
        }

    common_names = [
        name for name, _ in scores
        if name in weights
    ]

    total_weight = sum(
        weights[name]
        for name in common_names
    )

    ai_score = sum(
        score_dict[name] * weights[name]
        for name in common_names
    ) / total_weight

    label = "AI-generated" if ai_score >= threshold else "Human-written"

    distance = abs(ai_score - threshold)
    if distance < 0.05:
        confidence = "Uncertain"
    elif distance < 0.15:
        confidence = "Moderate"
    else:
        confidence = "Higher"

    return {
        "ai_score": float(ai_score),
        "ai_percentage": float(ai_score * 100),
        "label": label,
        "confidence": confidence,
        "threshold": float(threshold),
        "model_scores": score_dict,
        "similarity_percentage": None,
        "similarity_status": "Run separate similarity engine in Streamlit.",
    }

print("[OK] Section completed: 20. STREAMLIT-READY DOCUMENT-LEVEL SCORING FUNCTION")

[OK] Section completed: 20. STREAMLIT-READY DOCUMENT-LEVEL SCORING FUNCTION


## 21. SAVE MODELS

In [27]:

print_step("SAVING MODELS AND REPRODUCIBILITY FILES")

# ------------------------------------------------------------
# 1. Ensure output directories exist
# ------------------------------------------------------------

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Save the traditional ML models
# ------------------------------------------------------------

joblib.dump(
    lr_model,
    MODELS_DIR / "tfidf_logistic_regression.pkl",
)

joblib.dump(
    svm_model,
    MODELS_DIR / "tfidf_calibrated_linear_svm.pkl",
)

joblib.dump(
    adaptive_lr,
    MODELS_DIR / "adaptive_logistic_regression.pkl",
)

print("[OK] Logistic Regression saved.")
print("[OK] Calibrated Linear SVM saved.")
print("[OK] Adaptive Logistic Regression saved.")


# ------------------------------------------------------------
# 3. Save XGBoost if available
# ------------------------------------------------------------

if globals().get("xgb_model") is not None:
    joblib.dump(
        xgb_model,
        MODELS_DIR / "xgboost_linguistic.pkl",
    )
    print("[OK] XGBoost saved.")
else:
    print("[SKIP] XGBoost model is unavailable.")


# ------------------------------------------------------------
# 4. Save DistilRoBERTa if actually trained
#    No transformer_available variable is required.
# ------------------------------------------------------------

transformer_trainer = globals().get("trainer")
transformer_tokenizer = globals().get("tokenizer")

if (
    transformer_trainer is not None
    and transformer_tokenizer is not None
):
    transformer_dir = (
        MODELS_DIR / "distilroberta_texttrace"
    )
    transformer_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    transformer_trainer.save_model(
        str(transformer_dir)
    )

    transformer_tokenizer.save_pretrained(
        str(transformer_dir)
    )

    print("[OK] DistilRoBERTa model and tokenizer saved.")

else:
    print(
        "[SKIP] DistilRoBERTa was not saved because "
        "its trainer or tokenizer is unavailable."
    )


# ------------------------------------------------------------
# 5. Validate ensemble weights and model names
# ------------------------------------------------------------

if len(model_names) != len(best_weights):
    raise ValueError(
        "The number of ensemble model names does not match "
        "the number of ensemble weights."
    )

if len(model_names) == 0:
    raise ValueError(
        "No ensemble models are configured. "
        "Run the validation ensemble-search section first."
    )

if not np.isclose(np.sum(best_weights), 1.0, atol=1e-5):
    raise ValueError(
        f"Ensemble weights must sum to 1.0; "
        f"got {np.sum(best_weights):.6f}."
    )

if not 0.0 <= float(best_threshold) <= 1.0:
    raise ValueError(
        "The ensemble threshold must be between 0 and 1."
    )


# ------------------------------------------------------------
# 6. Save ensemble configuration
# ------------------------------------------------------------

ensemble_config = {
    "model_names": list(model_names),
    "weights": {
        name: float(weight)
        for name, weight in zip(
            model_names,
            best_weights,
        )
    },
    "threshold": float(best_threshold),
    "label_mapping": {
        "0": "Human",
        "1": "AI",
    },
    "seed": int(SEED),
    "dataset": "liamdugan/raid",
    "local_dataset": str(INPUT_CSV),
    "transformer_model": str(TRANSFORMER_MODEL),
    "transformer_train_limit": int(
        TRANSFORMER_TRAIN_LIMIT
    ),
}

with open(
    MODELS_DIR / "ensemble_config.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        ensemble_config,
        handle,
        indent=2,
    )

print("[OK] Ensemble configuration saved.")


# ------------------------------------------------------------
# 7. Save reproducibility metadata
# ------------------------------------------------------------

metadata = {
    "project": "TextTrace",
    "purpose": "AI-generated text detection",
    "dataset": "RAID",
    "dataset_configuration": "raid",
    "rows_after_cleaning": int(len(df)),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(validation_df)),
    "test_rows": int(len(test_df)),
    "seed": int(SEED),
    "domains": list(EXPECTED_DOMAINS),
    "models": list(model_names),
    "ensemble_threshold": float(best_threshold),
    "ensemble_weights": {
        name: float(weight)
        for name, weight in zip(
            model_names,
            best_weights,
        )
    },
    "transformer_saved": bool(
        transformer_trainer is not None
        and transformer_tokenizer is not None
    ),
    "note": (
        "AI score and source similarity are separate outputs. "
        "An AI score must not be described as proof of plagiarism "
        "or definitive proof of authorship."
    ),
}

with open(
    REPORTS_DIR / "reproducibility_metadata.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        metadata,
        handle,
        indent=2,
    )

print("[OK] Reproducibility metadata saved.")
print("\n[COMPLETE] Model-saving section finished.")



SAVING MODELS AND REPRODUCIBILITY FILES
[OK] Logistic Regression saved.
[OK] Calibrated Linear SVM saved.
[OK] Adaptive Logistic Regression saved.
[OK] XGBoost saved.


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

[OK] DistilRoBERTa model and tokenizer saved.
[OK] Ensemble configuration saved.
[OK] Reproducibility metadata saved.

[COMPLETE] Model-saving section finished.


## 22. FINAL SUMMARY

In [28]:
print_step("TEXTTRACE ML STAGE COMPLETE")

print("Final test ensemble:")
for key in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "FPR",
]:
    print(
        f"  {key:10s}: "
        f"{ensemble_test_metrics[key]:.4f}"
    )

print("\nSelected threshold:", best_threshold)
print("Selected weights:")
for name, weight in zip(
    model_names,
    best_weights,
):
    print(f"  {name:18s}: {weight:.4f}")

print("\nSaved outputs:")
for path in sorted(REPORTS_DIR.glob("*")):
    if path.is_file():
        print("  [OK]", path)

print("\nSaved charts:")
for path in sorted(CHARTS_DIR.glob("*.png")):
    print("  [OK]", path)

print("\nNext stage:")
print("  1. Build Streamlit document parser.")
print("  2. Add TXT/DOCX/PDF support.")
print("  3. Add sentence/paragraph/page location tracking.")
print("  4. Add separate public-web/local-repository similarity engine.")
print("  5. Add report generation.")
print("  6. Never label AI score as plagiarism proof.")
print("\nDONE.")

print("[OK] Section completed: 22. FINAL SUMMARY")


TEXTTRACE ML STAGE COMPLETE
Final test ensemble:
  Accuracy  : 0.9339
  Precision : 0.9420
  Recall    : 0.9618
  F1        : 0.9518
  ROC-AUC   : 0.9805
  FPR       : 0.1248

Selected threshold: 0.56
Selected weights:
  LR                : 0.3825
  SVM               : 0.0034
  XGBoost           : 0.3631
  DistilRoBERTa     : 0.2509

Saved outputs:
  [OK] E:\TexTrace1\reports\adaptive_comparison.csv
  [OK] E:\TexTrace1\reports\attack_results.csv
  [OK] E:\TexTrace1\reports\domain_distribution.csv
  [OK] E:\TexTrace1\reports\eda_text_length_summary.csv
  [OK] E:\TexTrace1\reports\error_analysis.csv
  [OK] E:\TexTrace1\reports\final_model_results.csv
  [OK] E:\TexTrace1\reports\label_distribution.csv
  [OK] E:\TexTrace1\reports\lr_xai_example.csv
  [OK] E:\TexTrace1\reports\README.txt
  [OK] E:\TexTrace1\reports\reproducibility_metadata.json
  [OK] E:\TexTrace1\reports\validation_model_results.csv
  [OK] E:\TexTrace1\reports\xgb_feature_importance.csv

Saved charts:
  [OK] E:\TexTrace1\